In [73]:
import pandas as pd
df =pd.read_excel("./1.xlsx",encoding='gbk')

In [74]:
#keywords=['25368281812226','21705688662597','22146630508597','21679671919597','21705688662597','21705688662597']
# #10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
#df['商品ID'].isin(keywords)
#df['商品ID'] = df['商品ID'].astype(str)


#删除其他行
#df.drop(index= df[ df['商品ID'].isin(keywords)==False].index,inplace=True)
#df.reset_index(drop=True,inplace= True)

In [75]:
df["实付款"] = df["实付款"].astype(str).str.replace("¥", "").astype(float)

In [76]:
from datetime import datetime
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单创建时间"] = df["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码"},inplace=True)
df1 = df1[["SKU编码","产品名称"]]
df= pd.merge(df,df1,on="SKU编码",how="left")

In [77]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码'].value_counts()
lackCode

Series([], Name: SKU编码, dtype: int64)

In [78]:
lackCode.to_excel('缺码表.xlsx')

In [79]:
df['订单数'] = df['成交数量']

In [80]:
df["下单单量"] = df['订单数']

In [81]:
# 清除未支付订单
from datetime import datetime
df["订单支付时间"] = df["订单支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["订单支付时间"] = pd.to_datetime(df["订单支付时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单支付时间"] = df["订单支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [82]:
df["发货时间"] = df["发货时间"].fillna(pd.to_datetime('1970-01-01'))
df["发货时间"] = pd.to_datetime(df["发货时间"],format = "%Y-%m-%d %H:%M:%S")
df["发货时间"] = df["发货时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [83]:
df["支付单量"] = 0
df["发货单量"] = 0
df["支付金额"] = 0
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01":
        df.loc[i,"发货单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'实付款']

In [84]:
df["售后状态"] = df["售后状态"].astype(str)

In [85]:
df["商品ID"] = df["商品ID"].astype(str)

In [86]:
df["售后状态"].value_counts()

nan      2525
退款成功      343
待卖家收货      27
退款关闭       12
待买家退货      12
Name: 售后状态, dtype: int64

In [87]:
df["商品ID"].value_counts()

21705688662597    1473
21679671919597     750
26369070333597     178
26369204448597     167
25791314991597     119
25780697089597      82
21707480339597      49
26833913412597      22
27141729734597      20
26362679698597      11
25789574330597       9
26061375807597       9
26176108581597       5
26853169840597       4
26060630775597       3
25034730950597       3
25010515646597       3
25781719748597       3
25006235672597       3
25950072514597       2
25777650137597       2
26263167491597       1
25790753253597       1
Name: 商品ID, dtype: int64

In [88]:
df['物流信息'].value_counts()

已签收      1766
未知        648
商家已接单     130
运输中       130
已发货        97
待取件        78
派送中        39
物流异常       25
已退签         6
Name: 物流信息, dtype: int64

In [89]:
for i in range (0,len(df)):
    if df.loc[i,"订单状态"] == "已收货" and (df.loc[i,'售后状态'] == 'nan' or df.loc[i,'售后状态'] == '退款关闭'):
        df.loc[i,"订单状态"] = "交易成功"

In [90]:
df["已签收"] = 0
df["发货中"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收数量"] = 0
df["发货前退款单量"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "交易成功" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"已签收"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已发货" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"发货中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"售后状态"] == "待买家处理" or df.loc[i,"售后状态"] == "待买家退货" or df.loc[i,"售后状态"] == "待卖家处理" or df.loc[i,"售后状态"]=="待卖家收货" or df.loc[i,"售后状态"]=="待平台判定":
        df.loc[i,"售后中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01"  and  df.loc[i,"发货时间"] == "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"预计签收单量"] = df.loc[i,"订单数"]

In [91]:
import numpy as np
result = pd.pivot_table(df,index=['订单创建时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额'] , aggfunc = np.sum)
result = result.reset_index()
result = result[['订单创建时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]

In [92]:
result.to_excel("111.xlsx",index=False)

In [93]:
# 拼接店铺消耗表
df_consume = pd.read_excel("快手店铺消耗9号.xlsx")
df_consume = df_consume[['日期','产品','花费','转化数']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['订单创建时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
df_consume.rename(columns={"产品":"产品名称"},inplace=True)
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['订单创建时间','产品名称'] , values = ['花费','转化数'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["订单创建时间"] = pd.to_datetime(df_consume["订单创建时间"],format = "%Y-%m-%d")
df_consume["订单创建时间"] = df_consume["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [94]:
final = pd.merge(result,df_consume,on = ['订单创建时间','产品名称'],how = 'left')

final = final[['订单创建时间',"产品名称",'转化数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]
final.rename(columns={"转化数":"总广告单量"},inplace=True)

In [95]:
final

,订单创建时间,产品名称,总广告单量,下单单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,支付金额
0,2026-10-22,8810内衣,63.0,101,77,73,64,2,0,7,4,66.0,3072.3
1,2026-10-23,8810内衣,113.0,170,131,117,104,0,0,13,14,104.0,5266.9
2,2026-10-24,8810内衣,112.0,158,127,125,112,1,0,12,2,113.0,5107.3
3,2026-10-25,8810内衣,109.0,168,124,112,93,1,0,18,12,94.0,4957.6
4,2026-10-26,8810内衣,100.0,146,113,107,104,0,0,3,6,104.0,4548.7
5,2026-10-27,8810内衣,104.0,166,124,120,99,0,2,19,4,99.0,4997.6
6,2026-10-28,8810内衣,92.0,139,114,110,90,2,1,17,4,92.0,4598.6
7,2026-10-29,8810内衣,106.0,143,114,105,87,0,0,18,9,87.0,4598.6
8,2026-10-30,8810内衣,116.0,173,132,126,110,1,2,13,6,111.0,5316.8
9,2026-10-31,8810内衣,101.0,142,108,103,86,1,0,16,5,87.0,4369.2


In [96]:
final.to_excel('直播签收汇总.xlsx')